In [4]:
!pip install ultralytics gradio opencv-python matplotlib -q

from ultralytics import YOLO
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import gradio as gr
import os
import pandas as pd
print("✅ Advanced Segmentation Setup Complete!")

✅ Advanced Segmentation Setup Complete!


# Day 13: Advanced Instance Segmentation Analysis

**What we will learn today:**
- Pixel-level Instance Segmentation using YOLO11-seg
- Area / Size Analysis of each object
- Color Analysis inside masks
- Saving individual masks as separate images
- Interactive Gradio Dashboard

This is a **much more advanced** skill compared to simple bounding box detection.

In [5]:
# Load Segmentation Model
model = YOLO("yolo11s-seg.pt")

def advanced_segmentation_analysis(image):
    results = model(image, conf=0.3, verbose=False)
    result = results[0]

    annotated = result.plot()
    analysis = []

    if result.masks is not None:
        for i, mask in enumerate(result.masks.data):
            mask_np = mask.cpu().numpy().astype(np.uint8)

            # Area Analysis
            area = np.sum(mask_np)

            # Color Analysis (mean color inside mask)
            if len(image.shape) == 3:
                masked_region = cv2.bitwise_and(image, image, mask=mask_np)
                mean_color = cv2.mean(masked_region, mask=mask_np)[:3]
                color_name = f"RGB({int(mean_color[0])}, {int(mean_color[1])}, {int(mean_color[2])})"
            else:
                color_name = "N/A"

            analysis.append({
                "Object": i+1,
                "Class": result.names[int(result.boxes.cls[i])],
                "Area (pixels)": area,
                "Approx Size": f"{area//1000}K pixels",
                "Mean Color": color_name
            })

            # Save individual mask
            mask_img = Image.fromarray(mask_np * 255)
            mask_img.save(f"mask_{i+1}.png")

    return annotated, analysis

In [11]:
# To address potential asyncio event loop issues in Colab
import nest_asyncio
nest_asyncio.apply()

In [15]:
def gradio_segmentation(image, confidence=0.3):
    # Convert PIL Image to NumPy array
    image_np = np.array(image)
    annotated, analysis = advanced_segmentation_analysis(image_np)
    df = pd.DataFrame(analysis)
    return annotated, df

with gr.Blocks(title="Advanced Segmentation Analyzer") as demo:
    gr.Markdown("# 🔬 Advanced Instance Segmentation Analysis")
    gr.Markdown("### Area, Color & Mask Analysis using YOLO11-seg")

    with gr.Row():
        input_img = gr.Image(type="pil", label="Upload Image")
        conf = gr.Slider(0.1, 0.95, 0.3, label="Confidence")

    btn = gr.Button("Analyze", variant="primary")

    with gr.Row():
        output_img = gr.Image(label="Segmentation Result")
        output_table = gr.DataFrame(label="Object Analysis")

    btn.click(
        fn=gradio_segmentation,
        inputs=[input_img, conf],
        outputs=[output_img, output_table]
    )

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://104dccb41c888da12a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7863 <> https://104dccb41c888da12a.gradio.live


In [8]:
# To address potential asyncio event loop issues in Colab
import nest_asyncio
nest_asyncio.apply()